# 04. Telegram connector

Добавляем реальный внешний сервис. Для воркшопа используем `dry_run=True`, чтобы показать payload без отправки сообщения.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'pyproject.toml').exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

DEFAULT_MODEL = 'openai:gpt-5.3-codex'

def model_name() -> str:
    return os.getenv('OPENCLAW_MODEL', DEFAULT_MODEL)


In [ ]:
BASE_PROMPT = """\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, and implementation.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.

If local shell execution is unavailable, use filesystem tools and clearly state
which verification would require shell access.
"""

In [ ]:
def workspace_root() -> Path:
    return Path(os.getenv("OPENCLAW_WORKSPACE", ".")).expanduser().resolve()


def backend():
    """Filesystem by default; local shell only when explicitly enabled."""
    root_dir = workspace_root()

    if os.getenv("OPENCLAW_ENABLE_LOCAL_SHELL") != "1":
        from deepagents.backends import FilesystemBackend

        return FilesystemBackend(root_dir=root_dir, virtual_mode=True)

    from deepagents.backends import LocalShellBackend

    return LocalShellBackend(
        root_dir=root_dir,
        virtual_mode=True,
        inherit_env=True,
        timeout=120,
        max_output_bytes=80_000,
    )

In [ ]:
from connectors.demo import DEMO_TOOLS
from connectors.telegram import TELEGRAM_TOOLS, send_telegram_message

print(
    send_telegram_message.invoke(
        {
            "message": "OpenClaw connectors are working.",
            "dry_run": True,
        }
    )
)

In [ ]:
agent = create_deep_agent(
    model=model_name(),
    tools=[*DEMO_TOOLS, *TELEGRAM_TOOLS],
    system_prompt=BASE_PROMPT,
    backend=backend(),
    interrupt_on={"execute": True},
)

In [ ]:
# Smoke check: the graph object is created locally.
print(type(agent).__name__)

Demo prompt: "Use the Telegram connector in dry-run mode. Prepare a short workshop message".